# Evidence Scout — Phase 1 Pipeline

## Table of Contents
1. [Section A — Setup](#Section-A-—-Setup)
2. [Section B — NLP Pipeline](#Section-B-—-NLP-Pipeline)
3. [Section C — CV Pipeline](#Section-C-—-CV-Pipeline)
4. [Section D — JSON Export](#Section-D-—-JSON-Export)
5. [Summary](#Summary)


## Section A — Setup
This cell mounts Google Drive to access the corpus and exports folders. It also installs the necessary dependencies, including Tesseract for OCR.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!apt install tesseract-ocr
!pip install nltk scikit-learn opencv-python pytesseract matplotlib pandas numpy

import nltk
nltk.download('stopwords')

import sys
import pandas as pd
import numpy as np
import cv2
import pytesseract
import sklearn

print("Setup complete")
print("Python version:", sys.version.split('|')[0])
print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)
print("OpenCV version:", cv2.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("PyTesseract version:", pytesseract.__version__)



## Section B — NLP Pipeline

### B1. Load corpus
This cell reads all markdown files from the corpus folder. It parses any frontmatter for the `title` and `source_url`. If frontmatter is missing, it falls back to the filename for the title.


In [ ]:
import os
import re
import pandas as pd

corpus_path = '/content/drive/MyDrive/evidence_scout/corpus/'

docs = []
if os.path.exists(corpus_path):
    for filename in os.listdir(corpus_path):
        if filename.endswith('.md'):
            with open(os.path.join(corpus_path, filename), 'r', encoding='utf-8') as f:
                content = f.read()
                
            title = filename[:-3].replace('_', ' ').title()
            source_url = ''
            full_text = content
            
            # Simple frontmatter parsing
            if content.startswith('---'):
                parts = content.split('---', 2)
                if len(parts) >= 3:
                    frontmatter = parts[1]
                    full_text = parts[2].strip()
                    
                    title_match = re.search(r'title:\s*(.*)', frontmatter)
                    if title_match:
                        title = title_match.group(1).strip()
                    
                    url_match = re.search(r'source_url:\s*(.*)', frontmatter)
                    if url_match:
                        source_url = url_match.group(1).strip()
                        
            docs.append({
                'filename': filename,
                'title': title,
                'source_url': source_url,
                'full_text': full_text,
                'clean_text': '' # Will populate in B2
            })

df = pd.DataFrame(docs)

if len(df) > 0:
    total_words = sum(len(text.split()) for text in df['full_text'])
    print(f"Number of documents loaded: {len(df)}")
    print(f"Total word count: {total_words}")
    print(f"Average words per document: {total_words / len(df):.1f}")
    display(df.head(3))
    
    # Show one example
    print("\n--- Example Document Metadata ---")
    row = df.iloc[0]
    print(f"Filename: {row['filename']}")
    print(f"Title: {row['title']}")
    print(f"URL: {row['source_url']}")
    print("Raw Content Start:")
    print(row['full_text'][:200], "...")
else:
    print("No documents found in corpus path.")



### B2. Preprocessing
This cell applies the strict preprocessing pipeline: markdown stripping, lowercase, punctuation removal, whitespace tokenization, and stopword removal. It intentionally excludes stemming and lemmatization to ensure perfect parity with the browser.


In [ ]:
import string
from nltk.corpus import stopwords
from collections import Counter
import re

# Preprocessing intentionally excludes stemming and lemmatization to match the browser pipeline exactly. See spec: CRITICAL CROSS-CUTTING REQUIREMENT.
standard_stops = set(stopwords.words('english'))
custom_stops = {"mg", "po", "bid", "tid", "qid", "patient", "doctor", "provider", "hospital", "tablet", "ml", "cc", "prn", "stat", "rn", "md", "np", "pa"}
all_stops = standard_stops.union(custom_stops)

def strip_markdown(text):
    text = re.sub(r'#+', '', text)
    text = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', text)
    text = re.sub(r'[*_`]', '', text)
    return text

def preprocess(text):
    # 1. Strip markdown
    t1 = strip_markdown(text)
    # 2. Lowercase
    t2 = t1.lower()
    # 3. Strip punctuation
    t3 = t2.translate(str.maketrans('', '', string.punctuation))
    # 4. Tokenize on whitespace
    t4 = t3.split()
    # 5. Remove stopwords
    t5 = [w for w in t4 if w not in all_stops]
    return t5

if len(df) > 0:
    # Representative document analysis
    idx = 0
    for i, t in enumerate(df['title']):
        if 'rotator cuff' in t.lower():
            idx = i
            break
            
    sample_text = df.iloc[idx]['full_text']
    print("--- BEFORE PREPROCESSING ---")
    print(sample_text[:500])
    
    t1 = strip_markdown(sample_text)
    t2 = t1.lower()
    t3 = t2.translate(str.maketrans('', '', string.punctuation))
    t4 = t3.split()
    t5 = [w for w in t4 if w not in all_stops]
    
    print("\n--- AFTER MARKDOWN STRIP (first 50) ---")
    print(t1.split()[:50])
    print("\n--- AFTER LOWERCASE (first 50) ---")
    print(t2.split()[:50])
    print("\n--- AFTER PUNCTUATION STRIP (first 50) ---")
    print(t3.split()[:50])
    print("\n--- AFTER TOKENIZATION (first 50) ---")
    print(t4[:50])
    print("\n--- AFTER STOPWORD REMOVAL (first 50) ---")
    print(t5[:50])
    
    # Frequencies
    all_raw_tokens = []
    all_clean_tokens = []
    
    for i, row in df.iterrows():
        raw_tokens = strip_markdown(row['full_text']).lower().translate(str.maketrans('', '', string.punctuation)).split()
        all_raw_tokens.extend(raw_tokens)
        
        clean = preprocess(row['full_text'])
        df.at[i, 'clean_text'] = ' '.join(clean)
        all_clean_tokens.extend(clean)
        
    print("\nTop 20 tokens BEFORE stopword removal:")
    print(Counter(all_raw_tokens).most_common(20))
    print("\nTop 20 tokens AFTER preprocessing:")
    print(Counter(all_clean_tokens).most_common(20))



### B3. TF-IDF vectorization
We use Scikit-Learn's TfidfVectorizer with our pre-tokenized inputs to generate term weights.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def identity(x):
    return x

if len(df) > 0:
    # We pass token lists directly
    token_lists = [text.split() for text in df['clean_text']]
    
    vectorizer = TfidfVectorizer(
        tokenizer=identity, 
        preprocessor=identity, 
        lowercase=False, 
        ngram_range=(1, 2)
    )
    
    tfidf_matrix = vectorizer.fit_transform(token_lists)
    vocab = vectorizer.get_feature_names_out()
    idfs = vectorizer.idf_
    
    print(f"Vocabulary size: {len(vocab)}")
    print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
    
    # Top 20 terms by IDF weight
    sorted_idf_idx = np.argsort(idfs)[::-1]
    print("\nTop 20 terms by IDF weight:")
    for idx in sorted_idf_idx[:20]:
        print(f"{vocab[idx]}: {idfs[idx]:.4f}")
        
    # Top 10 for rotator cuff document
    doc_idx = 0
    for i, t in enumerate(df['title']):
        if 'rotator cuff' in t.lower():
            doc_idx = i
            break
            
    doc_vector = tfidf_matrix[doc_idx].toarray()[0]
    sorted_tf_idx = np.argsort(doc_vector)[::-1]
    print(f"\nTop 10 TF-IDF terms for '{df.iloc[doc_idx]['title']}':")
    for idx in sorted_tf_idx[:10]:
        if doc_vector[idx] > 0:
            print(f"{vocab[idx]}: {doc_vector[idx]:.4f}")



### B4. Cosine similarity search function
This cell defines the core search engine that takes a query, preprocesses it, and computes cosine similarity against our documents.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def search(query, top_k=5):
    if len(df) == 0:
        return []
        
    q_tokens = preprocess(query)
    q_vec = vectorizer.transform([q_tokens])
    sims = cosine_similarity(q_vec, tfidf_matrix)[0]
    
    top_indices = np.argsort(sims)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        score = sims[idx]
        if score < 0.01: # simple threshold
            continue
            
        doc = df.iloc[idx]
        text = doc['full_text']
        
        # Try to find a matching term for the excerpt
        match_idx = -1
        for q_term in q_tokens:
            match_idx = text.lower().find(q_term)
            if match_idx != -1:
                break
                
        if match_idx != -1:
            start = max(0, match_idx - 100)
            end = min(len(text), match_idx + 100)
            excerpt = text[start:end]
            if start > 0: excerpt = '...' + excerpt
            if end < len(text): excerpt = excerpt + '...'
        else:
            excerpt = text[:200] + '...'
            
        results.append({
            'title': doc['title'],
            'url': doc['source_url'],
            'score': float(score),
            'excerpt': excerpt.replace('\n', ' ')
        })
        
    return results

if len(df) > 0:
    queries = [
        "safe positioning after rotator cuff repair",
        "elbow fracture immobilization",
        "pain management post op"
    ]
    
    for q in queries:
        print(f"\nQUERY: '{q}'")
        res = search(q)
        for r in res:
            print(f"- {r['score']:.4f} | {r['title']} | {r['excerpt']}")



### B5. Evaluation with MRR
Here we calculate the Mean Reciprocal Rank against a set of predefined test queries.


In [ ]:
test_queries = [
    {"query": "safe positioning after rotator cuff repair", "expected": "rotator cuff"},
    {"query": "immobilization phase elbow fracture", "expected": "elbow"},
    {"query": "shoulder dislocation protocol", "expected": "shoulder dislocation"},
    {"query": "biceps tenodesis restrictions", "expected": "biceps"},
    {"query": "ulnar nerve entrapment splinting", "expected": "ulnar"},
    {"query": "total shoulder arthroplasty precautions", "expected": "arthroplasty"},
    {"query": "olecranon bursitis wrap", "expected": "bursitis"},
    {"query": "epicondylitis strap placement", "expected": "epicondylitis"},
    {"query": "slap lesion repair rehab", "expected": "slap"},
    {"query": "frozen shoulder mobilization", "expected": "frozen"},
    {"query": "radial head fracture rom", "expected": "radial head"},
    {"query": "clavicle fracture sling duration", "expected": "clavicle"}
]

if len(df) > 0:
    results_list = []
    
    for tq in test_queries:
        q = tq["query"]
        exp = tq["expected"].lower()
        
        sims = cosine_similarity(vectorizer.transform([preprocess(q)]), tfidf_matrix)[0]
        sorted_idx = np.argsort(sims)[::-1]
        
        rank = -1
        actual_top = "None"
        
        if sims[sorted_idx[0]] > 0:
            actual_top = df.iloc[sorted_idx[0]]['title']
            
        for r, idx in enumerate(sorted_idx):
            if exp in df.iloc[idx]['title'].lower() and sims[idx] > 0:
                rank = r + 1
                break
                
        results_list.append({
            'query': q,
            'expected_document': exp,
            'actual_top_result': actual_top,
            'rank_of_expected': rank,
            'reciprocal_rank': 1/rank if rank > 0 else 0
        })
        
    mrr_df = pd.DataFrame(results_list)
    display(mrr_df)
    
    mrr = mrr_df['reciprocal_rank'].mean()
    print(f"\nFINAL MRR SCORE: {mrr:.4f}")



**MRR Interpretation**: The Mean Reciprocal Rank (MRR) evaluates the quality of our search engine. A score of 1.0 means the exact expected document is always returned as the very first result. Lower scores indicate the correct document appears lower down the list.


## Section C — CV Pipeline

### C1. Load sample image
We load a sample image containing text to test our OpenCV and Tesseract pipeline.


In [ ]:
import matplotlib.pyplot as plt
import cv2

sample_path = '/content/drive/MyDrive/evidence_scout/samples/sample1.jpg.jpeg'
image = None

if os.path.exists(sample_path):
    image = cv2.imread(sample_path)
else:
    # Create a dummy image for testing if the actual sample is missing
    print("Sample image not found, generating a dummy image with text for demonstration.")
    image = np.ones((500, 800, 3), dtype=np.uint8) * 255
    cv2.putText(image, "Safe positioning after rotator cuff repair", (50, 250), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,0), 2)
    
img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8,6))
plt.imshow(img_rgb)
plt.title("Original Image")
plt.axis('off')
plt.show()



### C2. Preprocessing pipeline
Convert to grayscale, apply adaptive and global thresholding, and clean up noise with morphological operations.


In [ ]:
# Grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
plt.figure(figsize=(8,6))
plt.imshow(gray, cmap='gray')
plt.title("Grayscale Image")
plt.axis('off')
plt.show()

# Adaptive Threshold
thresh_adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)

# Global Threshold
_, thresh_global = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16,6))
ax1.imshow(thresh_adaptive, cmap='gray')
ax1.set_title("Adaptive Thresholding")
ax1.axis('off')
ax2.imshow(thresh_global, cmap='gray')
ax2.set_title("Global Thresholding")
ax2.axis('off')
plt.show()

# Morphological cleanup (dilate then erode)
kernel = np.ones((3,3), np.uint8)
dilated = cv2.dilate(thresh_adaptive, kernel, iterations=1)
cleaned = cv2.erode(dilated, kernel, iterations=1)

plt.figure(figsize=(8,6))
plt.imshow(cleaned, cmap='gray')
plt.title("Morphologically Cleaned")
plt.axis('off')
plt.show()



### C3. Contour detection
Find the largest text area by detecting contours and filtering by area.


In [ ]:
contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print(f"Total contours detected: {len(contours)}")

# Filter
filtered_contours = [c for c in contours if cv2.contourArea(c) > 100]
filtered_contours = sorted(filtered_contours, key=cv2.contourArea, reverse=True)
print(f"Contours after filtering: {len(filtered_contours)}")

if len(filtered_contours) > 0:
    largest_contour = filtered_contours[0]
    print(f"Largest contour area: {cv2.contourArea(largest_contour)}")
    
    # Draw all contours
    img_all = img_rgb.copy()
    cv2.drawContours(img_all, filtered_contours, -1, (0, 255, 0), 2)
    plt.figure(figsize=(8,6))
    plt.imshow(img_all)
    plt.title("All Filtered Contours (Green)")
    plt.axis('off')
    plt.show()
    
    # Draw largest
    img_largest = img_rgb.copy()
    x, y, w, h = cv2.boundingRect(largest_contour)
    cv2.rectangle(img_largest, (x, y), (x+w, y+h), (255, 0, 0), 3)
    plt.figure(figsize=(8,6))
    plt.imshow(img_largest)
    plt.title("Largest Contour Bounding Box (Red)")
    plt.axis('off')
    plt.show()



### C4. Bitwise masking
Apply a mask based on the largest contour.


In [ ]:
if len(filtered_contours) > 0:
    mask = np.zeros(gray.shape, np.uint8)
    cv2.drawContours(mask, [largest_contour], -1, 255, thickness=cv2.FILLED)
    
    masked_img = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16,6))
    ax1.imshow(img_rgb)
    ax1.set_title("Original Image")
    ax1.axis('off')
    ax2.imshow(masked_img)
    ax2.set_title("Masked Image")
    ax2.axis('off')
    plt.show()



### C5. Crop and OCR
Crop to the bounding box and run PyTesseract to extract text and calculate average confidence.


In [ ]:
if len(filtered_contours) > 0:
    x, y, w, h = cv2.boundingRect(largest_contour)
    # Expand slightly
    margin = 10
    y1, y2 = max(0, y-margin), min(image.shape[0], y+h+margin)
    x1, x2 = max(0, x-margin), min(image.shape[1], x+w+margin)
    
    cropped = gray[y1:y2, x1:x2]
    
    plt.figure(figsize=(6,4))
    plt.imshow(cropped, cmap='gray')
    plt.title("Cropped for OCR")
    plt.axis('off')
    plt.show()
    
    text = pytesseract.image_to_string(cropped)
    data = pytesseract.image_to_data(cropped, output_type=pytesseract.Output.DICT)
    
    confidences = [int(conf) for conf in data['conf'] if int(conf) != -1]
    avg_conf = sum(confidences)/len(confidences) if confidences else 0
    
    print("--- OCR RESULT ---")
    print(text.strip() if text.strip() else "[No text detected]")
    print("------------------")
    print(f"Average OCR Confidence: {avg_conf:.1f}%")



### C6. End-to-end demo
Bring it all together: image -> OCR -> text -> search query.


In [ ]:
def cv_to_nlp_demo(img_path):
    print("Image loaded → ", end='')
    img = cv2.imread(img_path) if os.path.exists(img_path) else image
    if img is None:
        print("Failed to load image.")
        return
        
    print("Preprocessed → ", end='')
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    t = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    c, _ = cv2.findContours(t, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    valid_c = [cnt for cnt in c if cv2.contourArea(cnt) > 100]
    
    if not valid_c:
        print("No valid text regions found.")
        return
        
    largest = sorted(valid_c, key=cv2.contourArea, reverse=True)[0]
    x, y, w, h = cv2.boundingRect(largest)
    crop = g[max(0, y-10):y+h+10, max(0, x-10):x+w+10]
    
    text = pytesseract.image_to_string(crop).strip()
    print(f"Text extracted: [{text}] → Searching corpus → ")
    
    if text:
        print("\nTop results:")
        results = search(text)
        for r in results:
            print(f"- {r['score']:.4f} | {r['title']} | {r['excerpt']}")
    else:
        print("No text to search.")

# Run demo
cv_to_nlp_demo(sample_path)



## Section D — JSON Export
Export the vocabulary, IDF values, document vectors, and document metadata for use in the browser client.


In [ ]:
import json
import os

export_path = '/content/drive/MyDrive/evidence_scout/exports/'
os.makedirs(export_path, exist_ok=True)

if len(df) > 0:
    # 1. vocabulary.json
    vocab_dict = {term: int(idx) for term, idx in vectorizer.vocabulary_.items()}
    with open(os.path.join(export_path, 'vocabulary.json'), 'w', encoding='utf-8') as f:
        json.dump(vocab_dict, f)
        
    # 2. idf_values.json
    idf_dict = {term: float(idfs[vectorizer.vocabulary_[term]]) for term in vocab_dict}
    with open(os.path.join(export_path, 'idf_values.json'), 'w', encoding='utf-8') as f:
        json.dump(idf_dict, f)
        
    # 3. document_vectors.json
    doc_vecs = []
    for i in range(tfidf_matrix.shape[0]):
        row = tfidf_matrix.getrow(i)
        vec_dict = {str(col): float(val) for col, val in zip(row.indices, row.data)}
        doc_vecs.append(vec_dict)
    with open(os.path.join(export_path, 'document_vectors.json'), 'w', encoding='utf-8') as f:
        json.dump(doc_vecs, f)
        
    # 4. document_metadata.json
    metadata = []
    for i, row in df.iterrows():
        metadata.append({
            'id': i,
            'title': row['title'],
            'source_url': row['source_url'],
            'full_text': row['full_text'],
            'filename': row['filename']
        })
    with open(os.path.join(export_path, 'document_metadata.json'), 'w', encoding='utf-8') as f:
        json.dump(metadata, f)
        
    print(f"vocabulary.json: {os.path.getsize(os.path.join(export_path, 'vocabulary.json')) / 1024:.1f} KB")
    print(f"idf_values.json: {os.path.getsize(os.path.join(export_path, 'idf_values.json')) / 1024:.1f} KB")
    print(f"document_vectors.json: {os.path.getsize(os.path.join(export_path, 'document_vectors.json')) / 1024:.1f} KB")
    print(f"document_metadata.json: {os.path.getsize(os.path.join(export_path, 'document_metadata.json')) / 1024:.1f} KB")
    
    print("\nExport complete. 4 files written to Drive.")
else:
    print("No documents processed, skipping export.")



## Summary

**What was built:** This notebook demonstrates the full Natural Language Processing (NLP) and Computer Vision (CV) pipelines for Evidence Scout. It reads a corpus of Markdown files, preprocesses text, and computes TF-IDF representations. It also demonstrates how to load an image, isolate text regions via contour detection, and extract text via OCR (Tesseract).

**What each exported JSON contains:**
- `vocabulary.json`: Maps terms to their numeric index in the TF-IDF vector space.
- `idf_values.json`: Stores the Inverse Document Frequency weight for each term in the vocabulary.
- `document_vectors.json`: A sparse matrix representation (array of dicts) of each document's TF-IDF vector.
- `document_metadata.json`: Original text, titles, source URLs, and filenames for rendering in the UI.

**Usage in Web App:** The React web application will load these four JSON files locally. Instead of relying on a Python server to execute `cosine_similarity`, the web app will vectorize the user's search query locally and compare it directly to these precomputed `document_vectors` in the browser.

**Why Preprocessing is Simple:** Preprocessing intentionally uses simple lowercasing, punctuation stripping, and stopword removal without complex operations like stemming or lemmatization. This guarantees **perfect cross-platform parity** between this Python pipeline and the JavaScript client, ensuring the web app searches the exact same vocabulary index generated here.
